# Experiment 9 — Cross-Experiment Analysis and Comparative Interpretation

This notebook performs a comparative analysis across all experimental conditions investigated throughout the project.

The purpose of the analysis is to identify broader patterns regarding:
- preprocessing effectiveness,
- inductive bias,
- model robustness,
- representation quality,
- computational efficiency,
- and model complexity.

Rather than evaluating experiments individually, this notebook attempts to synthesize the findings across all degradation conditions and preprocessing levels in order to better understand how representation quality interacts with different machine learning paradigms.

Personally, I find this stage particularly interesting because isolated experiments often reveal only partial explanations. It is only when the results are viewed collectively that broader structural patterns begin to emerge regarding robustness, representation quality, and the practical trade-offs between simple and complex models.

## Project Path Configuration

The notebook explicitly adds the project root directory to the Python path in order to ensure that the centralized experimental framework within the `src/` directory can be imported consistently across notebooks.

In [ ]:
import sys
import os

PROJECT_ROOT = os.path.abspath("..")

if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

## Importing Analysis Libraries

The notebook imports the libraries required for loading experimental results, combining experiment outputs, and generating comparative visualizations.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path

## Loading Experimental Results

The previously generated experiment results are loaded from the centralized results directory structure.

In [ ]:
RESULTS_PATH = Path("../results/metrics")

In [ ]:
baseline_results = pd.read_csv(
    RESULTS_PATH / "baseline_results.csv"
)

preprocessing_results = pd.read_csv(
    RESULTS_PATH / "preprocessing_levels_results.csv"
)

missingness_results = pd.read_csv(
    RESULTS_PATH / "missingness_results.csv"
)

noise_results = pd.read_csv(
    RESULTS_PATH / "noise_results.csv"
)

feature_results = pd.read_csv(
    RESULTS_PATH / "feature_relevance_results.csv"
)

imbalance_results = pd.read_csv(
    RESULTS_PATH / "class_imbalance_results.csv"
)

dataset_results = pd.read_csv(
    RESULTS_PATH / "dataset_size_results.csv"
)

cost_results = pd.read_csv(
    RESULTS_PATH / "computational_cost_results.csv"
)

## Constructing Comparative Experiment Summaries

The experiments are summarized using mean F1-score values in order to support cross-experiment comparison.

In [ ]:
experiment_summary = pd.DataFrame({

    "Baseline":
        baseline_results.groupby("model")["f1_score"].mean(),

    "Preprocessing":
        preprocessing_results.groupby("model")["f1_score"].mean(),

    "Missingness":
        missingness_results.groupby("model")["f1_score"].mean(),

    "Noise":
        noise_results.groupby("model")["f1_score"].mean(),

    "Feature Relevance":
        feature_results.groupby("model")["f1_score"].mean(),

    "Class Imbalance":
        imbalance_results.groupby("model")["f1_score"].mean(),

    "Dataset Size":
        dataset_results.groupby("model")["f1_score"].mean()

})

experiment_summary

## Cross-Experiment Performance Visualization

The following figure compares average model performance across all experimental conditions.

In [ ]:
experiment_summary.T.plot(
    kind="bar",
    figsize=(14, 7)
)

plt.title(
    "Cross-Experiment F1-Score Comparison"
)

plt.ylabel("Mean F1-Score")

plt.xticks(rotation=0)

plt.tight_layout()

plt.show()

## Robustness Comparison

The following analysis examines how consistently models maintain predictive performance across degradation conditions.

In [ ]:
robustness_summary = (
    experiment_summary.std(axis=1)
    .sort_values()
    .to_frame(name="performance_variability")
)

robustness_summary

## Robustness Visualization

Lower variability indicates greater robustness across experimental conditions.

In [ ]:
robustness_summary.plot(
    kind="bar",
    figsize=(10, 6)
)

plt.title(
    "Model Robustness Across Experiments"
)

plt.ylabel(
    "Performance Variability"
)

plt.xticks(rotation=0)

plt.tight_layout()

plt.show()

## Representation Quality and Robustness Trends

The combined experiments reveal several broader trends regarding preprocessing effectiveness and representation quality.

Across nearly all degradation conditions, engineered preprocessing consistently improves predictive robustness relative to raw preprocessing conditions. This pattern appears especially strong for Logistic Regression, Naive Bayes, and the Multi-Layer Perceptron (MLP).

One particularly interesting observation is that preprocessing improvements frequently produce performance gains comparable to — or greater than — increases in model complexity alone. In several experiments, comparatively simple classical models become highly competitive once representation quality improves sufficiently.

At the same time, more flexible neural architectures still maintain advantages under several degraded conditions, particularly when the representation space becomes highly noisy or structurally unstable.

In [ ]:
preprocessing_comparison = (
    preprocessing_results
    .groupby(["preprocessing", "model"])["f1_score"]
    .mean()
    .unstack()
)

preprocessing_comparison.plot(
    kind="bar",
    figsize=(12, 6)
)

plt.title(
    "Representation Quality Effects Across Models"
)

plt.ylabel("Mean F1-Score")

plt.xticks(rotation=0)

plt.tight_layout()

plt.show()